In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
pd.set_option('display.max_columns', None)

In [ ]:
# Define the base path
base_path = "./data-all-waves"
file_path = os.path.join(base_path, "df_final.csv")
df_final = pd.read_csv(file_path)

In [ ]:
# 1. clean EDMOM and EDDAD

In [ ]:
# Function to clean EDMOM and EDDAD values
def clean_ed(x):
    if pd.isna(x):  # Keep NaN as is
        return x
    if isinstance(x, str):
        if x.isdigit():  # Convert string numbers to int
            return int(x)
        elif x == 'E':   # Keep 'E' as is
            return 'E'
    return x  # Return original value if it's already an integer

# Apply the cleaning function to EDMOM and EDDAD for all years
df_final['EDMOM'] = df_final['EDMOM'].apply(clean_ed)
df_final['EDDAD'] = df_final['EDDAD'].apply(clean_ed)

# # Verify the cleaned values
# for year in sorted(df_final['REFYR'].unique()):
#     edmom_values = df_final[df_final['REFYR'] == year]['EDMOM'].unique()
#     eddad_values = df_final[df_final['REFYR'] == year]['EDDAD'].unique()
    
#     print(f"\nREFYR {year}:")
#     print(sorted([x for x in edmom_values if x != 'E']) + (['E'] if 'E' in edmom_values else []))
#     print(sorted([x for x in eddad_values if x != 'E']) + (['E'] if 'E' in eddad_values else []))


In [ ]:
# 2. Merge with UR data

import os
import pandas as pd

# Define the path and read the CSV file
base_path = "./data-all-waves"
ur_ba_path = os.path.join(base_path, "ur_ba_16y.csv")

# Read the CSV file
ur_ba_16y = pd.read_csv(ur_ba_path)

# Rename columns
ur_ba_16y = ur_ba_16y.rename(columns={'CGBD16O': 'UR'})

# First prepare yearly unemployment rates for both degree types
# For Bachelor's
ur_ba_16y['observation_date'] = pd.to_datetime(ur_ba_16y['observation_date'])
ur_ba_16y['year'] = ur_ba_16y['observation_date'].dt.year


# Assume there is a 'YEAR' column in ur_ba_16y indicating the year
# If not, check if 'BAYR' corresponds to a year column
if 'year' in ur_ba_16y.columns:
    # Compute the annual average unemployment rate
    ur_avg = ur_ba_16y.groupby('year')['UR'].mean().reset_index()
    
    # Rename YEAR to BAYR so it matches df_final
    ur_avg = ur_avg.rename(columns={'year': 'BAYR'})
else:
    print("Error: No 'year' column found in ur_ba_16y. Please check the dataset structure.")

# Merge with df_final based on BAYR
df_final= df_final.merge(ur_avg, on='BAYR', how='left')

In [ ]:
# 3. Create log salary
df_final['log_salary'] = np.log(df_final['SALARY'])

# 4. Create experience variables
df_final['exp'] = df_final['REFYR'] - df_final['BAYR']
df_final['exp2'] = df_final['exp']**2

# 5. Create dummy for master's degree
df_final['master'] = (df_final['DGRDG'] == 2).astype(int)

In [ ]:
# 6. Remove salary outliers for each REFYR

# Function to get valid salary range for each year
def get_valid_salary_range(data, year):
   year_data = data[data['REFYR'] == year]
   p1 = year_data['SALARY'].quantile(0.01)
   p99 = year_data['SALARY'].quantile(0.99)
   return p1, p99

# # Print sample sizes before trimming
# print("Before trimming:")
# print(df_final.groupby('REFYR').size())

# Remove outliers by year
clean_salary_dfs = []
for year in df_final['REFYR'].unique():
   year_data = df_final[df_final['REFYR'] == year].copy()
   p1, p99 = get_valid_salary_range(df_final, year)
#    print(f"\nYear {year}:")
#    print(f"1st percentile: ${p1:,.2f}")
#    print(f"99th percentile: ${p99:,.2f}")
   clean_year_data = year_data[(year_data['SALARY'] >= p1) & (year_data['SALARY'] <= p99)]
   clean_salary_dfs.append(clean_year_data)

# Combine cleaned data
df_clean_salary = pd.concat(clean_salary_dfs)

In [ ]:
# 7. restrict to exp [3,14]

df = df_clean_salary[df_clean_salary['exp'].between(3, 14)].copy()


In [ ]:
df.columns

In [ ]:
len(df)

In [ ]:
df['BAYR'].max()

# Sample Sizes

In [ ]:
import pandas as pd
import numpy as np

# Group by graduation year (BAYR) and master's status
cohort_counts = df.groupby(['BAYR', 'master']).size().unstack(fill_value=0)

# Rename the columns
if 0 in cohort_counts.columns and 1 in cohort_counts.columns:
    cohort_counts.columns = ["Frequency (only with Bachelor's)", "Frequency (with Master's)"]

# Calculate average unemployment rate by cohort year
ur_by_cohort = df.groupby('BAYR')['gradur_ba'].mean().round(2)
cohort_counts['National UR'] = ur_by_cohort

# Add a total row
cohort_counts.loc['Total'] = [
    cohort_counts["Frequency (only with Bachelor's)"].sum(),
    cohort_counts["Frequency (with Master's)"].sum(),
    np.nan  # No meaningful total for unemployment rate
]

# Reset index to make BAYR a column
cohort_counts = cohort_counts.reset_index().rename(columns={'BAYR': 'Cohort'})

# Format the Cohort column as integer (except for the 'Total' row)
cohort_counts['Cohort'] = cohort_counts['Cohort'].apply(
    lambda x: int(x) if isinstance(x, (int, float)) else x
)

# Display the table
print(cohort_counts)

# Save to CSV
cohort_counts.to_csv('cohort_master_counts.csv', index=False)

# Optional: Create a more detailed version with percentages
detailed_counts = cohort_counts.copy()
non_total_rows = detailed_counts['Cohort'] != 'Total'

# Calculate percentages for non-total rows
if "Frequency (only with Bachelor's)" in detailed_counts.columns and "Frequency (with Master's)" in detailed_counts.columns:
    total_by_year = detailed_counts.loc[non_total_rows, "Frequency (only with Bachelor's)"] + \
                    detailed_counts.loc[non_total_rows, "Frequency (with Master's)"]
    
    detailed_counts.loc[non_total_rows, "Bachelor's %"] = (
        100 * detailed_counts.loc[non_total_rows, "Frequency (only with Bachelor's)"] / total_by_year
    ).round(1)
    
    detailed_counts.loc[non_total_rows, "Master's %"] = (
        100 * detailed_counts.loc[non_total_rows, "Frequency (with Master's)"] / total_by_year
    ).round(1)

# Display the detailed table
print("\nDetailed table with percentages:")
print(detailed_counts)

# Save the detailed table
detailed_counts.to_csv('cohort_master_counts_detailed.csv', index=False)

# Alternative approach for counting unique individuals
unique_bachelors = df[df['master'] == 0]['assigned_id'].nunique()
unique_masters = df[df['master'] == 1]['assigned_id'].nunique()

print(f"Unique individuals with only Bachelor's: {unique_bachelors}")
print(f"Unique individuals with Master's: {unique_masters}")

In [ ]:
import pandas as pd
import numpy as np

def create_balance_table(df, cont_vars, cat_vars, group_var=None):
    """
    Create a balance table for continuous and categorical variables
    
    Parameters:
    -----------
    df : pandas DataFrame
        Input data
    cont_vars : list
        List of continuous variables
    cat_vars : list
        List of categorical variables
    group_var : str, optional
        Variable to group by (e.g., 'master')
    
    Returns:
    --------
    pandas DataFrame
        Balance table
    """
    results = []
    
    # Function to calculate stats for a continuous variable
    def get_cont_stats(data, var):
        return {
            'Variable': var,
            'Type': 'Continuous',
            'N': data[var].count(),
            'Mean': data[var].mean(),
            'Std': data[var].std(),
            'Min': data[var].min(),
            'Median': data[var].median(),
            'Max': data[var].max()
        }
    
    # Function to calculate stats for a categorical variable
    def get_cat_stats(data, var):
        value_counts = data[var].value_counts(dropna=False)
        total = len(data)
        
        # Create one row per category
        cat_results = []
        for cat, count in value_counts.items():
            cat_name = f"{var} = {cat}"
            cat_results.append({
                'Variable': cat_name,
                'Type': 'Categorical',
                'N': count,
                '% of Total': 100 * count / total,
                'Mean': None,
                'Std': None,
                'Min': None,
                'Median': None,
                'Max': None
            })
        return cat_results
    
    # If no grouping, calculate stats for the entire dataset
    if group_var is None:
        # Continuous variables
        for var in cont_vars:
            if var in df.columns:
                results.append(get_cont_stats(df, var))
        
        # Categorical variables
        for var in cat_vars:
            if var in df.columns:
                results.extend(get_cat_stats(df, var))
    
    # If grouping, calculate stats for each group
    else:
        groups = df[group_var].unique()
        
        for group in groups:
            group_data = df[df[group_var] == group]
            group_prefix = f"{group_var}={group}"
            
            # Continuous variables
            for var in cont_vars:
                if var in df.columns:
                    stats = get_cont_stats(group_data, var)
                    stats['Variable'] = f"{group_prefix}: {var}"
                    results.append(stats)
            
            # Categorical variables
            for var in cat_vars:
                if var in df.columns and var != group_var:
                    cat_stats = get_cat_stats(group_data, var)
                    for stat in cat_stats:
                        stat['Variable'] = f"{group_prefix}: {stat['Variable']}"
                    results.extend(cat_stats)
    
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    # Reorder columns
    cols = ['Variable', 'Type', 'N']
    if '% of Total' in results_df.columns:
        cols.append('% of Total')
    cols.extend(['Mean', 'Std', 'Min', 'Median', 'Max'])
    
    # Only include columns that exist
    cols = [col for col in cols if col in results_df.columns]
    
    return results_df[cols]

# Example usage:
# Define continuous and categorical variables
continuous_vars = ['REFYR', 'BAYR', 'exp', 'UR', 'SALARY', 'AGE']
categorical_vars = ['master', 'GENDER', 'RACEM', 'EDMOM', 'EDDAD']

# Create balance table for the entire dataset
balance_table = create_balance_table(df, continuous_vars, categorical_vars)
print(balance_table)

# Create balance table grouped by master's degree status
balance_table_by_master = create_balance_table(df, continuous_vars, 
                                              [var for var in categorical_vars if var != 'master'], 
                                              group_var='master')
print(balance_table_by_master)

# For a cleaner presentation, you might want to format the numeric columns
def format_balance_table(table):
    format_dict = {
        'Mean': '{:.2f}',
        'Std': '{:.2f}',
        'Min': '{:.2f}',
        'Median': '{:.2f}',
        'Max': '{:.2f}',
        '% of Total': '{:.1f}%'
    }
    
    for col, fmt in format_dict.items():
        if col in table.columns:
            table[col] = table[col].apply(lambda x: fmt.format(x) if pd.notnull(x) else '')
    
    return table

# Format the tables
formatted_table = format_balance_table(balance_table)
formatted_table_by_master = format_balance_table(balance_table_by_master)

In [ ]:
# Save to HTML
formatted_table.to_html('balance_table.html', index=False, na_rep='')

In [ ]:
import pandas as pd
import numpy as np

def create_balance_table_continuous(df, cont_vars, group_var=None):
    """
    Create a balance table for continuous variables only
    
    Parameters:
    -----------
    df : pandas DataFrame
        Input data
    cont_vars : list
        List of continuous variables
    group_var : str, optional
        Variable to group by (e.g., 'master')
    
    Returns:
    --------
    pandas DataFrame
        Balance table for continuous variables
    """
    results = []
    
    # Function to calculate stats for a continuous variable
    def get_cont_stats(data, var):
        return {
            'Variable': var,
            'N': data[var].count(),
            'Mean': data[var].mean(),
            'Std': data[var].std(),
            'Min': data[var].min(),
            'Median': data[var].median(),
            'Max': data[var].max()
        }
    
    # If no grouping, calculate stats for the entire dataset
    if group_var is None:
        # Continuous variables
        for var in cont_vars:
            if var in df.columns:
                results.append(get_cont_stats(df, var))
    
    # If grouping, calculate stats for each group
    else:
        groups = df[group_var].unique()
        
        for group in sorted(groups):
            group_data = df[df[group_var] == group]
            
            # Continuous variables
            for var in cont_vars:
                if var in df.columns:
                    stats = get_cont_stats(group_data, var)
                    stats['Variable'] = f"{var}"
                    stats['Group'] = group
                    results.append(stats)
    
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    # Reorder columns for ungrouped table
    if group_var is None:
        cols = ['Variable', 'N', 'Mean', 'Std', 'Min', 'Median', 'Max']
    else:
        # For grouped table, show Group column first
        cols = ['Variable', 'Group', 'N', 'Mean', 'Std', 'Min', 'Median', 'Max']
    
    # Only include columns that exist
    cols = [col for col in cols if col in results_df.columns]
    
    return results_df[cols]

# Example usage:
# Define continuous variables
continuous_vars = ['REFYR', 'BAYR', 'exp', 'UR', 'SALARY', 'AGE']

# Create balance table for the entire dataset
balance_table = create_balance_table_continuous(df, continuous_vars)
print(balance_table)

# Create balance table grouped by master's degree status
balance_table_by_master = create_balance_table_continuous(df, continuous_vars, group_var='master')
print(balance_table_by_master)

# For a cleaner presentation, format the numeric columns
def format_balance_table(table):
    format_dict = {
        'Mean': '{:.2f}',
        'Std': '{:.2f}',
        'Min': '{:.2f}',
        'Median': '{:.2f}',
        'Max': '{:.2f}'
    }
    
    for col, fmt in format_dict.items():
        if col in table.columns:
            table[col] = table[col].apply(lambda x: fmt.format(x) if pd.notnull(x) else '')
    
    return table

# Format the tables
formatted_table = format_balance_table(balance_table)
formatted_table_by_master = format_balance_table(balance_table_by_master)

# # If you want to save to a CSV or Excel for further formatting
# formatted_table.to_csv('balance_table_continuous.csv', index=False)
# formatted_table_by_master.to_csv('balance_table_continuous_by_master.csv', index=False)

In [ ]:
# Alternative: Create a side-by-side comparison table for master's vs non-master's
def create_side_by_side_table(df, cont_vars, group_var='master'):
    # Get the balance table with groups
    balance_table = create_balance_table_continuous(df, cont_vars, group_var)
    
    # Reshape to have groups as columns
    groups = sorted(df[group_var].unique())
    
    reshaped_tables = []
    for stat in ['N', 'Mean', 'Std', 'Min', 'Median', 'Max']:
        pivot = balance_table.pivot(index='Variable', columns='Group', values=stat)
        pivot.columns = [f"{stat}_{col}" for col in pivot.columns]
        reshaped_tables.append(pivot)
    
    # Combine all stats
    result = pd.concat(reshaped_tables, axis=1)
    
    # Reorder columns to group by master status
    ordered_cols = []
    for group in groups:
        for stat in ['N', 'Mean', 'Std', 'Min', 'Median', 'Max']:
            ordered_cols.append(f"{stat}_{group}")
    
    return result[ordered_cols]

# Create side-by-side table
side_by_side_table = create_side_by_side_table(df, continuous_vars)
print(side_by_side_table)

# Format and save
side_by_side_table.to_csv('balance_table_side_by_side.csv')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Count how many times each assigned_id appears in the dataset
id_counts = df['assigned_id'].value_counts()

# Create a frequency summary of these counts
frequency_summary = id_counts.value_counts().sort_index()

# Display the summary table
print("Frequency Summary of assigned_id:")
print("--------------------------------")
print("Times appeared | Number of people")
print("--------------------------------")
for times, count in frequency_summary.items():
    print(f"{times:14d} | {count:16d}")
print("--------------------------------")
print(f"Total unique IDs: {len(id_counts)}")

# Calculate percentages
total_ids = len(id_counts)
percentages = (frequency_summary / total_ids) * 100

# Create a more detailed summary DataFrame
summary_df = pd.DataFrame({
    'Times_Appeared': frequency_summary.index,
    'Number_of_People': frequency_summary.values,
    'Percentage': percentages.values
})

# Display the DataFrame
print("\nDetailed Summary:")
print(summary_df)

# Optional: Create a bar chart visualization
plt.figure(figsize=(10, 6))
plt.bar(summary_df['Times_Appeared'], summary_df['Number_of_People'])
plt.xlabel('Number of Times Person Appears in Dataset')
plt.ylabel('Count of People')
plt.title('Frequency Distribution of Appearances in Dataset')
plt.xticks(summary_df['Times_Appeared'])
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add value labels on top of each bar
for i, v in enumerate(summary_df['Number_of_People']):
    plt.text(i+1, v + 0.1, str(v), ha='center')

plt.tight_layout()
plt.savefig('id_frequency_distribution.png')
plt.close()

# You can also calculate cumulative statistics
summary_df['Cumulative_Count'] = summary_df['Number_of_People'].cumsum()
summary_df['Cumulative_Percentage'] = summary_df['Cumulative_Count'] / total_ids * 100

print("\nCumulative Summary:")
print(summary_df[['Times_Appeared', 'Number_of_People', 'Percentage', 
                  'Cumulative_Count', 'Cumulative_Percentage']])

# Save the summary to a CSV file for reference
summary_df.to_csv('assigned_id_frequency_summary.csv', index=False)

In [ ]:
import pandas as pd

# First, let's check for duplicated assigned_ids within each REFYR
duplicate_counts_by_year = {}

# Get the unique reference years
ref_years = sorted(df['REFYR'].unique())

# For each reference year, check for duplicate assigned_ids
for year in ref_years:
    # Subset the data for this year
    year_data = df[df['REFYR'] == year]
    
    # Count occurrences of each assigned_id in this year
    id_counts = year_data['assigned_id'].value_counts()
    
    # Find ids that appear more than once
    duplicates = id_counts[id_counts > 1]
    
    # Store the count of such ids
    duplicate_counts_by_year[year] = len(duplicates)
    
    # Optionally, store the actual duplicated ids
    duplicated_ids = duplicates.index.tolist()

# Print the summary
print("Number of assigned_ids appearing more than once in the same reference year:")
print("------------------------------------------------------------------------")
print("Reference Year | Count of Duplicated IDs")
print("------------------------------------------------------------------------")
for year, count in duplicate_counts_by_year.items():
    print(f"{year:14d} | {count:22d}")
print("------------------------------------------------------------------------")

# Create a DataFrame for easier analysis
duplicate_summary = pd.DataFrame({
    'REFYR': list(duplicate_counts_by_year.keys()),
    'Number_of_Duplicated_IDs': list(duplicate_counts_by_year.values())
})

# Calculate the total number of duplicated IDs across all years
total_duplicates = sum(duplicate_counts_by_year.values())
print(f"\nTotal number of assigned_ids appearing more than once in the same reference year: {total_duplicates}")

# Optionally, get more detailed information about the duplications
detailed_duplicates = []
for year in ref_years:
    year_data = df[df['REFYR'] == year]
    duplicated_in_year = year_data['assigned_id'].value_counts()
    
    for id_val, count in duplicated_in_year[duplicated_in_year > 1].items():
        detailed_duplicates.append({
            'REFYR': year,
            'assigned_id': id_val,
            'appearance_count': count
        })

# Convert to DataFrame if there are any duplicates
if detailed_duplicates:
    detailed_df = pd.DataFrame(detailed_duplicates)
    print("\nSample of detailed duplication information:")
    print(detailed_df.head(10))  # Show first 10 duplicated cases
    
    # Save detailed information to CSV
    detailed_df.to_csv('detailed_duplicated_ids_by_year.csv', index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Create a cross-tabulation of experience level and master status
exp_master_counts = pd.crosstab(df['exp'], df['master'])

# Rename the columns for clarity
exp_master_counts.columns = ['Bachelor\'s Only', 'Master\'s']

# Calculate the total for each experience level
exp_master_counts['Total'] = exp_master_counts.sum(axis=1)

# Calculate percentages
exp_master_counts['Bachelor\'s %'] = (exp_master_counts['Bachelor\'s Only'] / exp_master_counts['Total'] * 100).round(1)
exp_master_counts['Master\'s %'] = (exp_master_counts['Master\'s'] / exp_master_counts['Total'] * 100).round(1)

# Display the results
print("Counts by Experience Level and Master's Status:")
print(exp_master_counts)

# Save to CSV
exp_master_counts.to_csv('exp_master_counts.csv')

# Create a stacked bar chart
plt.figure(figsize=(12, 6))
exp_master_counts[['Bachelor\'s Only', 'Master\'s']].plot(kind='bar', stacked=True, figsize=(12, 6))
plt.title('Number of Observations by Experience Level and Degree Type')
plt.xlabel('Years of Experience')
plt.ylabel('Number of Observations')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Degree Type')
plt.tight_layout()
plt.savefig('exp_master_distribution.png')
plt.close()

# Calculate the overall proportion
total_bachelors = exp_master_counts['Bachelor\'s Only'].sum()
total_masters = exp_master_counts['Master\'s'].sum()
total_observations = total_bachelors + total_masters

print(f"\nOverall Summary:")
print(f"Total Bachelor's Only: {total_bachelors} ({total_bachelors/total_observations*100:.1f}%)")
print(f"Total Master's: {total_masters} ({total_masters/total_observations*100:.1f}%)")
print(f"Total Observations: {total_observations}")